In [1]:
import sys, os
sys.path.append(os.path.abspath("../../"))

# Load shared utilities for attention-from-scratch analysis
from scripts.attention_from_scratch_utils import *

# Set reproducible seed
set_seed(42)

print("Environment ready.")
print("Device:", device)

Environment ready.
Device: mps


# **06 — Minimal Transformer Encoder**

Now that we’ve built all the core components individually — attention, multi-head projections, positional encodings, feed-forward networks, layer normalization, and residuals — this notebook brings **everything together** into a fully working Transformer encoder layer.

---

## **What This Notebook Covers**

### **1. Build the End-to-End Encoder Layer**
- Combine token embeddings with positional information  
- Apply multi-head self-attention  
- Add residual connections and layer normalization  
- Pass results through the position-wise feed-forward network  

### **2. Wire Components Into the Standard Transformer Block**
- Follow the exact architecture used in real models  
- Ensure tensor dimensions match across all sublayers  
- Implement the full forward pass of an encoder block  

### **3. Run and Inspect the Encoder**
- Feed simple sequences through the encoder  
- Print and visualize intermediate tensors  
- Examine attention weights, head outputs, and post-FFN activations  

---

## **Outcome**
By the end of this notebook, you’ll have a **fully functional Transformer encoder layer**, built entirely from scratch using only the pieces developed in earlier notebooks.  
This serves as the foundation for constructing deeper encoders and full Transformer models in later sections.

# **Minimal Transformer Encoder — High-Level ASCII Architecture**

                ┌──────────────────────────────┐
                │        Input Tokens          │
                └──────────────────────────────┘
                               │
                               ▼
                ┌──────────────────────────────┐
                │   Token + Positional Embed   │
                └──────────────────────────────┘
                               │
                               ▼
        ┌─────────────────────────────────────────────────┐
        │                Encoder Layer                    │
        │                                                 │
        │   ┌─────────────────────────────────────────┐   │
        │   │        Multi-Head Self-Attention        │   │
        │   └─────────────────────────────────────────┘   │
        │                         │                       │
        │                         ▼                       │
        │   ┌─────────────────────────────────────────┐   │
        │   │      Add & Norm (after attention)       │   │
        │   └─────────────────────────────────────────┘   │
        │                         │                       │
        │                         ▼                       │
        │   ┌─────────────────────────────────────────┐   │
        │   │     Feed-Forward Network (FFN)          │   │
        │   └─────────────────────────────────────────┘   │
        │                         │                       │
        │                         ▼                       │
        │   ┌─────────────────────────────────────────┐   │
        │   │       Add & Norm (after FFN)            │   │
        │   └─────────────────────────────────────────┘   │
        └─────────────────────────────────────────────────┘
                               │
                               ▼
                ┌──────────────────────────────┐
                │         Encoder Output        │
                └──────────────────────────────┘

# **Step 1: Start With the Inputs — Token + Positional Embeddings**

The very first thing we need to do in this notebook is to build the **input representation** for the encoder.  
Before attention, before FFNs, before residuals — the model must turn raw token IDs into vectors it can process.

### What we build first:
1. **Token Embedding Layer**  
   Converts input token IDs (e.g., `[12, 4, 88, 5]`) into dense vectors.

2. **Positional Encoding / Positional Embedding**  
   Adds information about sequence order so the encoder can tell *which token is where*. We will follow a more modern approach where we use learned embeddings for PE
   

4. **Combine them:**  
   ```text
   X = TokenEmbedding(input_ids) + PositionalEncoding

In [53]:
import torch
import torch.nn as nn
import pandas as pd

# ---------------------------------------------------------
# Token Embedding
# ---------------------------------------------------------

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x)


# ---------------------------------------------------------
# Learned Positional Embedding
# ---------------------------------------------------------

class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_emb = nn.Embedding(max_len, d_model)

    def forward(self, x):
        # x: (batch_size, seq_len)
        batch_size, seq_len = x.size()
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)  # (1, seq_len)
        return self.pos_emb(positions)  # (1, seq_len, d_model)


# ---------------------------------------------------------
# Example Setup
# ---------------------------------------------------------

vocab_size = 50
d_model = 8
batch_size = 1
seq_len = 4
max_len = 100

torch.manual_seed(0)

token_emb = TokenEmbedding(vocab_size, d_model)
pos_emb = LearnedPositionalEmbedding(max_len, d_model)

# Sample input IDs
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

# Compute embeddings
token_vectors = token_emb(input_ids)[0].detach()
pos_vectors = pos_emb(input_ids)[0].detach()
combined_vectors = token_vectors + pos_vectors

# ---------------------------------------------------------
# Convert to DataFrames
# ---------------------------------------------------------

df_token = pd.DataFrame(token_vectors.numpy(), columns=[f"d{i}" for i in range(d_model)])
df_pos = pd.DataFrame(pos_vectors.numpy(), columns=[f"d{i}" for i in range(d_model)])
df_combined = pd.DataFrame(combined_vectors.numpy(), columns=[f"d{i}" for i in range(d_model)])

# Display results
print("Token Embeddings:\n", df_token, "\n")
print("Learned Positional Embeddings:\n", df_pos, "\n")
print("Combined Embeddings (Token + Position):\n", df_combined)

Token Embeddings:
        d0      d1      d2      d3      d4      d5      d6      d7
0 -0.0933  0.6871 -0.8383  0.0009  0.8419 -0.4000  1.0395  0.3582
1  0.5411  1.5390  1.0860  1.2464  0.1151  1.6193  0.4637  1.3007
2 -0.1420 -0.5341 -0.5234  0.8615 -0.8870  0.8388  1.1529 -1.7611
3 -0.1420 -0.5341 -0.5234  0.8615 -0.8870  0.8388  1.1529 -1.7611 

Learned Positional Embeddings:
        d0      d1      d2      d3      d4      d5      d6      d7
0 -0.7645  0.2408  0.1664 -2.2318  1.3892 -0.5023  1.6797 -1.0240
1  1.6859 -1.2177  0.7650  1.1971 -0.7128 -0.0656  2.2050  1.7852
2 -0.0118  0.9797 -1.0661  1.7720 -0.2793 -0.2769  0.7489 -0.6435
3 -0.9518  0.2715  0.6716  1.8500  1.1910 -0.5899  0.9647 -1.5094 

Combined Embeddings (Token + Position):
        d0      d1      d2      d3      d4      d5      d6      d7
0 -0.8578  0.9279 -0.6719 -2.2309  2.2311 -0.9024  2.7192 -0.6658
1  2.2271  0.3213  1.8510  2.4435 -0.5977  1.5537  2.6687  3.0859
2 -0.1539  0.4455 -1.5894  2.6335 -1.1662  0.5

# **Next Step: Apply LayerNorm (Pre-Norm) Before Multi-Head Self-Attention**

Now that we have the combined input embeddings  

X = TokenEmbedding + PositionalEmbedding

the next step—since you want to follow **pre-norm architecture**—is to **normalize X before feeding it into Multi-Head Self-Attention (MHA).**

---

## What Pre-Norm Means

In a **pre-norm Transformer**, each sublayer begins with a LayerNorm:

X → LayerNorm → Sublayer (MHA or FFN) → Add (Residual)

This is different from the original 2017 Transformer (post-norm):

Sublayer → Add → LayerNorm

But pre-norm is now the dominant architecture in modern models  
(GPT-2, GPT-3, LLaMA, T5, etc.) because it improves:

- training stability  
- gradient flow  
- ability to support deep networks  

---

## What We Implement Next

We introduce a **LayerNorm layer** and apply it to the embeddings before MHA:

X_norm = LayerNorm(X)
attention_output = MHA(X_norm)

Then add the residual connection:

---

## Summary of This Step

1. Instantiate `nn.LayerNorm(d_model)`  
2. Apply it to the combined embeddings  
3. The **normalized embeddings** become the input to Multi-Head Self-Attention  
4. Residual connection happens afterward  

---

## Goal

After this step, you'll have the **input pipeline for the MHA block ready**, and we can move on to implementing the attention mechanism itself.

In [54]:
import torch
import torch.nn as nn
import pandas as pd

# ---------------------------------------------------------
# Token Embedding
# ---------------------------------------------------------

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x)


# ---------------------------------------------------------
# Learned Positional Embedding
# ---------------------------------------------------------

class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_emb = nn.Embedding(max_len, d_model)

    def forward(self, x):
        batch_size, seq_len = x.size()
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        return self.pos_emb(positions)


# ---------------------------------------------------------
# Pre-Norm Before Multi-Head Attention
# ---------------------------------------------------------

class PreNormMHAInput(nn.Module):
    def __init__(self, vocab_size, d_model, max_len):
        super().__init__()
        self.token_emb = TokenEmbedding(vocab_size, d_model)
        self.pos_emb = LearnedPositionalEmbedding(max_len, d_model)
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, input_ids):
        # 1. Token embeddings
        tok = self.token_emb(input_ids)  # (batch, seq, d_model)

        # 2. Positional embeddings
        pos = self.pos_emb(input_ids)    # (batch, seq, d_model)

        # 3. Combine
        x = tok + pos

        # 4. Pre-norm (this is the new step)
        x_norm = self.layernorm(x)

        return tok, pos, x, x_norm


# ---------------------------------------------------------
# Example Setup
# ---------------------------------------------------------

vocab_size = 50
d_model = 8
batch_size = 1
seq_len = 4
max_len = 100

torch.manual_seed(0)

model = PreNormMHAInput(vocab_size, d_model, max_len)

# Random token IDs
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

tok, pos, combined, normed = model(input_ids)

# ---------------------------------------------------------
# Convert to DataFrames for visualization
# ---------------------------------------------------------

df_tok = pd.DataFrame(tok[0].detach().numpy(), columns=[f"d{i}" for i in range(d_model)])
df_pos = pd.DataFrame(pos[0].detach().numpy(), columns=[f"d{i}" for i in range(d_model)])
df_combined = pd.DataFrame(combined[0].detach().numpy(), columns=[f"d{i}" for i in range(d_model)])
df_normed = pd.DataFrame(normed[0].detach().numpy(), columns=[f"d{i}" for i in range(d_model)])

print("Token Embeddings:\n", df_tok, "\n")
print("Positional Embeddings:\n", df_pos, "\n")
print("Combined Embeddings:\n", df_combined, "\n")
print("LayerNorm Output (Pre-MHA):\n", df_normed)

Token Embeddings:
        d0      d1      d2      d3      d4      d5      d6      d7
0 -0.0933  0.6871 -0.8383  0.0009  0.8419 -0.4000  1.0395  0.3582
1  0.5411  1.5390  1.0860  1.2464  0.1151  1.6193  0.4637  1.3007
2 -0.1420 -0.5341 -0.5234  0.8615 -0.8870  0.8388  1.1529 -1.7611
3 -0.1420 -0.5341 -0.5234  0.8615 -0.8870  0.8388  1.1529 -1.7611 

Positional Embeddings:
        d0      d1      d2      d3      d4      d5      d6      d7
0 -0.7645  0.2408  0.1664 -2.2318  1.3892 -0.5023  1.6797 -1.0240
1  1.6859 -1.2177  0.7650  1.1971 -0.7128 -0.0656  2.2050  1.7852
2 -0.0118  0.9797 -1.0661  1.7720 -0.2793 -0.2769  0.7489 -0.6435
3 -0.9518  0.2715  0.6716  1.8500  1.1910 -0.5899  0.9647 -1.5094 

Combined Embeddings:
        d0      d1      d2      d3      d4      d5      d6      d7
0 -0.8578  0.9279 -0.6719 -2.2309  2.2311 -0.9024  2.7192 -0.6658
1  2.2271  0.3213  1.8510  2.4435 -0.5977  1.5537  2.6687  3.0859
2 -0.1539  0.4455 -1.5894  2.6335 -1.1662  0.5619  1.9018 -2.4046
3 -1.09

# **Next Step: Build Multi-Head Self-Attention (MHA)**

You've now completed:

1. **Token embeddings**
2. **Learned positional embeddings**
3. **Combined embeddings**
4. **LayerNorm (pre-norm)**

The next layer in the Transformer encoder pipeline is:

# **Multi-Head Self-Attention (MHA)**

This is where the model actually learns how tokens relate to each other.

---

# **What we build next (in order)**

## **1. Linear projections for Q, K, V**
From the normalized embeddings `X_norm`, we compute:

Q = X_norm @ W_Q
K = X_norm @ W_K
V = X_norm @ W_V

Each projection is its own `nn.Linear`.

This transforms the input into:
- queries (what each token wants to look for)
- keys (what each token offers)
- values (the information carried)

---

## **2. Split Q, K, V into multiple heads**
For example:

- `d_model = 64`
- `num_heads = 8`
- `d_head = 8`

We reshape:

(batch, seq, d_model)
→ (batch, heads, seq, d_head)

This lets each head focus on different patterns in the sequence.

---

## **3. Scaled Dot-Product Attention**
For each head:

scores = (Q · Kᵀ) / sqrt(d_head)
weights = softmax(scores)
attention = weights @ V

This is the actual “attention” calculation.

---

## **4. Concatenate the heads**
Recombine the heads:

(batch, heads, seq, d_head)
→ (batch, seq, d_model)

---

## **5. Output projection**
Final linear transform:

output = ConcatHeads @ W_O

This ensures the output stays in `d_model` space so it can connect to the next layers.

---

# **After MHA**
When MHA is done, we will:

1. Add the **residual connection**
2. Apply **LayerNorm** again
3. Move on to the **Feed-Forward Network (FFN)**

But for now, the next coding step is:

# **Implement the Multi-Head Self-Attention module**


In [56]:
import torch
import torch.nn as nn
import math
import pandas as pd

# ---------------------------------------------------------
# Helper: Convert (seq_len, d_model) tensors to DataFrames
# ---------------------------------------------------------
def to_df(tensor, prefix="d"):
    arr = tensor.detach().numpy()
    cols = [f"{prefix}{i}" for i in range(arr.shape[-1])]
    return pd.DataFrame(arr, columns=cols)


# ---------------------------------------------------------
# Step-by-step Multi-Head Self-Attention
# (aligned exactly with your numbered explanation)
# ---------------------------------------------------------

class StepByStepMHA(nn.Module):
    def __init__(self, d_model=8, num_heads=2):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must divide num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        # 1. Linear projections for Q, K, V
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)

        # 5. Output projection
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    # Utility: split tensor into heads
    def split_heads(self, x):
        batch, seq, d_model = x.size()
        x = x.view(batch, seq, self.num_heads, self.d_head)
        return x.permute(0, 2, 1, 3)    # (batch, heads, seq, d_head)

    def forward(self, X_norm):
        results = {}

        # ---------------------------------------------------------
        # 1. Linear projections (Q, K, V)
        # ---------------------------------------------------------
        Q = self.W_Q(X_norm)
        K = self.W_K(X_norm)
        V = self.W_V(X_norm)

        results["Q"] = Q[0]
        results["K"] = K[0]
        results["V"] = V[0]

        # ---------------------------------------------------------
        # 2. Split into multiple heads
        # ---------------------------------------------------------
        Qh = self.split_heads(Q)
        Kh = self.split_heads(K)
        Vh = self.split_heads(V)

        results["Q_heads"] = Qh[0]
        results["K_heads"] = Kh[0]
        results["V_heads"] = Vh[0]

        # ---------------------------------------------------------
        # 3. Scaled Dot-Product Attention
        # ---------------------------------------------------------
        scores = torch.matmul(Qh, Kh.transpose(-2, -1)) / math.sqrt(self.d_head)
        weights = torch.softmax(scores, dim=-1)
        attention = torch.matmul(weights, Vh)

        results["scores"] = scores[0]
        results["weights"] = weights[0]
        results["attention"] = attention[0]

        # ---------------------------------------------------------
        # 4. Concatenate heads
        # ---------------------------------------------------------
        attention = attention.permute(0, 2, 1, 3).contiguous()
        concat = attention.view(X_norm.size(0), X_norm.size(1), self.d_model)

        results["concat"] = concat[0]

        # ---------------------------------------------------------
        # 5. Final linear projection
        # ---------------------------------------------------------
        output = self.W_O(concat)
        results["final_output"] = output[0]

        return results


# ---------------------------------------------------------
# Example Input (pretend this is LayerNorm output)
# ---------------------------------------------------------

torch.manual_seed(0)

batch_size = 1
seq_len = 4
d_model = 8
num_heads = 2

X_norm = torch.randn(batch_size, seq_len, d_model)

mha = StepByStepMHA(d_model=d_model, num_heads=num_heads)
results = mha(X_norm)

# ---------------------------------------------------------
# Build Tables (each step)
# ---------------------------------------------------------

tables = {}

tables["Q"] = to_df(results["Q"], "Q")
tables["K"] = to_df(results["K"], "K")
tables["V"] = to_df(results["V"], "V")

# Per-head Q, K, V
for h in range(num_heads):
    tables[f"Q_head_{h}"] = pd.DataFrame(
        results["Q_heads"][h].detach().numpy(),
        columns=[f"h{h}_d{i}" for i in range(mha.d_head)]
    )
    tables[f"K_head_{h}"] = pd.DataFrame(
        results["K_heads"][h].detach().numpy(),
        columns=[f"h{h}_d{i}" for i in range(mha.d_head)]
    )
    tables[f"V_head_{h}"] = pd.DataFrame(
        results["V_heads"][h].detach().numpy(),
        columns=[f"h{h}_d{i}" for i in range(mha.d_head)]
    )

# Attention scores & weights
for h in range(num_heads):
    tables[f"scores_head_{h}"] = pd.DataFrame(
        results["scores"][h].detach().numpy()
    )
    tables[f"weights_head_{h}"] = pd.DataFrame(
        results["weights"][h].detach().numpy()
    )

# Attention output before concatenation
att = results["attention"].reshape(seq_len, -1).detach().numpy()
tables["head_attention_concat"] = pd.DataFrame(
    att,
    columns=[f"h{h}_d{i}" for h in range(num_heads) for i in range(mha.d_head)]
)

# After concatenation
tables["concat"] = to_df(results["concat"], "c")

# Final output
tables["final_output"] = to_df(results["final_output"], "o")

# ---------------------------------------------------------
# Print all tables so reader follows the flow easily
# ---------------------------------------------------------

for name, df in tables.items():
    print(f"\n===== {name} =====\n")
    print(df)


===== Q =====

       Q0      Q1      Q2      Q3      Q4      Q5      Q6      Q7
0  0.2740 -0.5357  0.4996  0.4568  0.5752 -0.4019  0.4703  0.8802
1  0.4285  0.0959  0.5452  0.1603  0.3231 -0.4026 -0.6084  0.2687
2 -0.1492  0.4926  0.1838  0.2361 -0.6579 -0.1636 -0.3624  0.0270
3  0.1055 -0.5587  0.5858  0.3784  0.8792 -0.2359 -0.4635  0.6587

===== K =====

       K0      K1      K2      K3      K4      K5      K6      K7
0  0.5575  0.1937 -0.3161 -0.2506  0.2481  0.2962  0.8905  0.2799
1 -0.0113 -0.4279 -0.4356 -0.3471  0.5648  0.8352 -0.4592 -0.6702
2 -0.9657  0.4430 -0.2921 -0.3547  0.4126 -0.0532  0.7246 -0.0719
3  0.8688 -0.5097 -0.5648 -0.5898  0.0252  1.2148 -0.9176 -0.3453

===== V =====

       V0      V1      V2      V3      V4      V5      V6      V7
0  0.0939  0.3577 -0.0266 -0.9967  0.4012 -0.7007 -0.2774  0.7368
1 -0.3526 -0.2229  0.3676  0.3002 -0.3560 -0.3855 -0.1463  0.7265
2  0.0492  1.0481  0.4909 -0.6687 -0.2387 -0.6625 -0.0693  0.0383
3 -0.3454 -0.1491  0.0513  0

## **Next Step: LayerNorm Before the Feed-Forward Network (FFN)**

Since we're building a **pre-norm Transformer encoder**, every sublayer begins with a LayerNorm.  
You've already applied LayerNorm before MHA — now we do the same before the FFN.

After computing the MHA output and adding the residual:

residual_1 = X + MHA(X_norm)

we apply:

residual_1_norm = LayerNorm(residual_1)

This normalized tensor becomes the input to the Feed-Forward Network.

### **Why?**
Pre-norm architecture improves stability and gradient flow, especially in deeper models.  
So the FFN always receives a normalized input.

In [60]:
import torch
import torch.nn as nn
import pandas as pd

# ---------------------------------------------------------
# Fake example values to simulate real Transformer flow
# (X = input to encoder, mha_output = result of MHA block)
# ---------------------------------------------------------

batch_size = 1
seq_len = 4
d_model = 8

torch.manual_seed(0)

X = torch.randn(batch_size, seq_len, d_model)          # original input
mha_output = torch.randn(batch_size, seq_len, d_model) # output from MHA

# ---------------------------------------------------------
# 1. Residual connection: X + MHA(X_norm)
# ---------------------------------------------------------

residual_1 = X + mha_output

# ---------------------------------------------------------
# 2. LayerNorm BEFORE the Feed-Forward Network (Pre-Norm)
# ---------------------------------------------------------

layernorm = nn.LayerNorm(d_model)
residual_1_norm = layernorm(residual_1)

# ---------------------------------------------------------
# Convert to tables for nicer viewing
# ---------------------------------------------------------

df_residual = pd.DataFrame(
    residual_1[0].detach().numpy(),
    columns=[f"d{i}" for i in range(d_model)]
)

df_normed = pd.DataFrame(
    residual_1_norm[0].detach().numpy(),
    columns=[f"d{i}" for i in range(d_model)]
)

print("\n===== Residual After MHA (X + MHA_output) =====\n")
print(df_residual)

print("\n===== LayerNorm Applied (Pre-FFN Input) =====\n")
print(df_normed)


===== Residual After MHA (X + MHA_output) =====

       d0      d1      d2      d3      d4      d5      d6      d7
0 -1.7394 -1.1208 -0.7433 -0.1855  1.2884  0.8044  0.3248 -1.6741
1  0.2200 -0.4709  0.0603  0.3606  0.6427  3.5399 -0.3521 -1.8340
2 -2.0257 -0.8231  1.6220  0.9714  0.3685 -1.9468  0.2019  1.4578
3  0.3040  0.1585  1.3476  3.5940 -0.1418  0.3522  2.7660 -1.3952

===== LayerNorm Applied (Pre-FFN Input) =====

       d0      d1      d2      d3      d4      d5      d6      d7
0 -1.2865 -0.7007 -0.3433  0.1848  1.5803  1.1220  0.6679 -1.2246
1 -0.0357 -0.5204 -0.1477  0.0630  0.2609  2.2938 -0.4371 -1.4769
2 -1.4914 -0.5964  1.2233  0.7391  0.2904 -1.4327  0.1665  1.1012
3 -0.3744 -0.4700  0.3120  1.7896 -0.6676 -0.3426  1.2450 -1.4920


# **Next Step: Build the Feed-Forward Network (FFN)**

You now have:

1. **X** → original encoder input  
2. **MHA output**  
3. **Residual connection:** `residual_1 = X + mha_output`  
4. **Pre-FFN LayerNorm:** `residual_1_norm = LayerNorm(residual_1)`  

The next component in the Transformer encoder is:

# **Feed-Forward Network (FFN)**

This is the **second major sublayer** in the encoder block, and it operates **independently on each token**, position by position.

---

## **What the FFN looks like**

A standard Transformer FFN is:

FFN(x) = Linear1(x) → Activation → Linear2(x)

Where:

- `Linear1` expands the dimensionality (e.g., 512 → 2048)
- Activation is typically **ReLU** or **GELU**
- `Linear2` projects back down to `d_model`

So:

d_model  →  d_ff  →  d_model

Example:

8 → 32 → 8

---

## **After computing FFN(residual_1_norm)**

We then add the **second residual connection**:

output = residual_1 + FFN(residual_1_norm)

This completes **one full encoder layer**.


In [63]:
import torch
import torch.nn as nn
import pandas as pd

# ---------------------------------------------------------
# Starting from your previous step:
# df_normed = LayerNorm(X + MHA_output)
# We will convert df_normed back to a tensor and apply the FFN.
# ---------------------------------------------------------

# Fake example values again (same setup as before)
batch_size = 1
seq_len = 4
d_model = 8
d_ff = 32   # Feed-forward expansion size

torch.manual_seed(0)

# Fake inputs for X and MHA output
X = torch.randn(batch_size, seq_len, d_model)
mha_output = torch.randn(batch_size, seq_len, d_model)

# Residual
residual_1 = X + mha_output

# Pre-FFN LayerNorm
layernorm = nn.LayerNorm(d_model)
residual_1_norm = layernorm(residual_1)

# Convert pre-FFN norm to DataFrame (your df_normed)
df_normed = pd.DataFrame(
    residual_1_norm[0].detach().numpy(),
    columns=[f"d{i}" for i in range(d_model)]
)

print("\n===== Pre-FFN LayerNorm (df_normed) =====\n")
print(df_normed)


# ---------------------------------------------------------
# Convert df_normed back to a tensor so we can run FFN
# ---------------------------------------------------------

x_ffn_input = torch.tensor(df_normed.values, dtype=torch.float32).unsqueeze(0)
# shape: (1, seq_len, d_model)


# ---------------------------------------------------------
# 3-step Feed-Forward Network (FFN)
# ---------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.activation = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # Step 1: First Linear layer
        out1 = self.linear1(x)

        # Step 2: Activation (ReLU)
        out2 = self.activation(out1)

        # Step 3: Final Linear layer back to d_model
        out3 = self.linear2(out2)

        return out1, out2, out3


ffn = FeedForward(d_model, d_ff)

# Run FFN on the normalized input
ffn_lin1, ffn_act, ffn_lin2 = ffn(x_ffn_input)

# ---------------------------------------------------------
# Convert each FFN step to DataFrames for clean visualization
# ---------------------------------------------------------

df_ffn_lin1 = pd.DataFrame(ffn_lin1[0].detach().numpy(), columns=[f"ff1_{i}" for i in range(d_ff)])
df_ffn_act  = pd.DataFrame(ffn_act[0].detach().numpy(),  columns=[f"act_{i}" for i in range(d_ff)])
df_ffn_lin2 = pd.DataFrame(ffn_lin2[0].detach().numpy(), columns=[f"ff2_{i}" for i in range(d_model)])

# ---------------------------------------------------------
# Display step-by-step results
# ---------------------------------------------------------

print("\n===== FFN Step 1: Linear(d_model → d_ff) =====\n")
print(df_ffn_lin1)

print("\n===== FFN Step 2: Activation (ReLU) =====\n")
print(df_ffn_act)

print("\n===== FFN Step 3: Linear(d_ff → d_model) =====\n")
print(df_ffn_lin2)


===== Pre-FFN LayerNorm (df_normed) =====

       d0      d1      d2      d3      d4      d5      d6      d7
0 -1.2865 -0.7007 -0.3433  0.1848  1.5803  1.1220  0.6679 -1.2246
1 -0.0357 -0.5204 -0.1477  0.0630  0.2609  2.2938 -0.4371 -1.4769
2 -1.4914 -0.5964  1.2233  0.7391  0.2904 -1.4327  0.1665  1.1012
3 -0.3744 -0.4700  0.3120  1.7896 -0.6676 -0.3426  1.2450 -1.4920

===== FFN Step 1: Linear(d_model → d_ff) =====

    ff1_0   ff1_1   ff1_2   ff1_3   ff1_4   ff1_5   ff1_6   ff1_7   ff1_8   ff1_9  ff1_10  ff1_11  ff1_12  ff1_13  ff1_14  ff1_15  ff1_16  ff1_17  ff1_18  ff1_19  ff1_20  ff1_21  ff1_22  ff1_23  ff1_24  ff1_25  ff1_26  ff1_27  ff1_28  ff1_29  ff1_30  ff1_31
0  0.5258 -0.3290 -0.0346  0.5836  0.8218  0.4725 -0.0679 -0.6751  0.3963  0.8904 -0.2873 -0.3126  0.1888  0.6783 -0.2065 -0.5065  0.3051 -0.2754  0.0239  1.0339 -0.3424 -0.7756  0.4262  0.6725 -0.1601  0.9478  0.6386  0.7085  0.3717 -0.5211  0.0227 -0.0180
1  1.1954 -0.5135 -0.3465  0.9850  1.0600  0.2262 -0.1388 -0.

## **Final Step of the Encoder Block: Second Residual Connection**

After passing the normalized tensor through the Feed-Forward Network (FFN),  
we obtain the final FFN output:

ffn_out = FFN(residual_1_norm)

In a **pre-norm Transformer encoder**, every sublayer ends with a **residual connection**.

So we take the output from the **previous residual** and **add** the FFN output:

final_output = residual_1 + ffn_out

Where:

- `residual_1 = X + MHA(X_norm)`  
- `residual_1_norm = LayerNorm(residual_1)`  
- `ffn_out = FFN(residual_1_norm)`  

This completes **one full Transformer encoder layer**, producing the layer’s output.

---

### Why this matters
The residual connection:

- stabilizes gradients  
- preserves input information  
- allows deeper networks to train effectively  
- is a core part of Transformer architecture  

Now let's implement this final step.

In [64]:
import torch
import torch.nn as nn
import pandas as pd

# ---------------------------------------------------------
# Fake example values (same as before)
# ---------------------------------------------------------

batch_size = 1
seq_len = 4
d_model = 8
d_ff = 32

torch.manual_seed(0)

# Fake inputs for X and MHA output
X = torch.randn(batch_size, seq_len, d_model)
mha_output = torch.randn(batch_size, seq_len, d_model)

# ---------------------------------------------------------
# 1. First residual: X + MHA(X_norm)
# ---------------------------------------------------------

residual_1 = X + mha_output

# ---------------------------------------------------------
# 2. LayerNorm BEFORE FFN
# ---------------------------------------------------------

layernorm = nn.LayerNorm(d_model)
residual_1_norm = layernorm(residual_1)

# ---------------------------------------------------------
# 3. Feed-Forward Network
# ---------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.activation = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        out1 = self.linear1(x)
        out2 = self.activation(out1)
        out3 = self.linear2(out2)
        return out3  # final FFN output

ffn = FeedForward(d_model, d_ff)

ffn_out = ffn(residual_1_norm)

# ---------------------------------------------------------
# 4. Final residual connection: output = residual_1 + ffn_out
# ---------------------------------------------------------

final_output = residual_1 + ffn_out

# Convert to DataFrame for clean viewing
df_final = pd.DataFrame(
    final_output[0].detach().numpy(),
    columns=[f"d{i}" for i in range(d_model)]
)

print("\n===== Final Encoder Output (After Second Residual) =====\n")
print(df_final)


===== Final Encoder Output (After Second Residual) =====

       d0      d1      d2      d3      d4      d5      d6      d7
0 -1.5581 -0.7436 -1.0723 -0.1926  1.5694  0.5397  0.6011 -1.3636
1  0.4728 -0.0091  0.0055  0.3083  0.7818  3.3461 -0.2111 -1.4423
2 -1.8735 -0.9148  1.5133  0.6070  0.6702 -2.0413  0.6198  1.5128
3  0.2138  0.2581  1.4636  3.5083  0.1689  0.0439  3.1519 -1.2982


# Milestone Reached: You Now Understand the Inner Workings of a Transformer Encoder

You’ve just completed one of the most important steps in modern deep learning —  
building a **Transformer encoder layer from scratch**.

This isn’t just “writing some code.”  
You reconstructed the core architecture that powers:

- **GPT models**
- **BERT**
- **T5**
- **LLaMA**
- **Modern vision transformers**
- …and essentially every state-of-the-art model in NLP today.

---

## What You Now Understand

You’ve walked through every moving part of a Transformer encoder:

### **1. Token + Positional Embeddings**
How raw token IDs become continuous, position-aware vectors.

### **2. Layer Normalization (Pre-Norm)**
Why normalization before sublayers stabilizes deep networks.

### **3. Multi-Head Self-Attention**
How each token attends to every other token and builds contextual meaning.

### **4. Residual Pathways**
How skip connections preserve information and smooth gradient flow.

### **5. Feed-Forward Networks (FFN)**
Why each position gets a high-capacity MLP to refine its representation.

### **6. The Two-Sublayer Structure**

LN → MHA → Residual
LN → FFN → Residual

The heartbeat of the Transformer block.

---

## Why This Is a Big Deal

You didn’t just use a library function.  
You built the Transformer layer — the same architecture used in billion-parameter models — piece by piece, tensor by tensor, step by step.

You now understand:

- **what every tensor means**
- **why each operation exists**
- **how information flows through the block**
- **how attention and FFNs transform representations**
- **how residual + norm keep everything stable**

This depth of understanding is rare, and it’s the foundation for all advanced Transformer work.

---

## What This Unlocks Next

Because you understand the internals, you can now:

- Build your own **EncoderLayer** module  
- Stack layers to create a **full Transformer encoder**  
- Feed real text and inspect attention maps  
- Train your own miniature Transformer  
- Explore variants like ALiBi, Rotary embeddings, SwiGLU FFN, etc.  
- Understand and modify state-of-the-art model architectures

# How Do We Get From **“One Transformer Layer”** to **ChatGPT Answering Questions?**

This is *the* question — and the moment where the small pieces click into the big picture.

Right now, you’ve built:

- embeddings  
- attention  
- FFN  
- residuals  
- normalization  

That’s **one encoder layer** — the fundamental *building block* of a Transformer.

But ChatGPT is not just a layer.  
ChatGPT is a **stack of dozens of these layers**, trained on *mind-boggling amounts of text*.

Let’s walk through how we get from your layer → ChatGPT answering questions.

---

# 1. **Stack Many Layers**
A GPT-like model has:

- 12 layers (small models)
- 24 layers (medium)
- 96+ layers (GPT-4 scale)

Each layer refines the representation from the previous one.

Input → Layer 1 → Layer 2 → Layer 3 → … → Layer N

Every layer does:
- attention (what should I focus on?)
- FFN (how do I transform this meaning?)
- residual pathways (don’t lose information)

The output becomes a deeply learned representation of the text.

---

# 2. **Add a Language Modeling Head**

At the very top, we add a simple linear layer:

hidden_state → Linear(d_model → vocab_size)

This produces a probability distribution for **the next token**.

That’s it.

You train the model to predict the next word:

“The cat sat on the” → model should predict “mat”

This is called **autoregressive language modeling**.

---

# 3. **Train on Massive Text**
ChatGPT was trained on:

- books  
- articles  
- websites  
- code  
- conversations  
- datasets curated to teach reasoning, math, science, social context, etc.

During training, the model repeatedly tries to guess the next token and adjusts its weights to improve.

After trillions of predictions, it develops:

- grammar
- facts
- style
- reasoning patterns
- ability to stay on topic
- ability to answer questions

Because all of these things *help it predict the next token better*.

---

# 4. **Add Chat Instruction Tuning**
Raw GPT models are good at language modeling but not great at behaving like assistants.

So we fine-tune them on:

- human-written Q&A examples  
- instruction-following datasets  
- preference data (RLHF)  

The model learns:

- how to follow instructions  
- how to answer questions  
- how to refuse unsafe queries  
- how to format its output  
- how to hold a conversation  

This tuning process makes it **helpful**, **harmless**, and **aligned**.

---

# 5. **During Inference: How ChatGPT Answers You**
When you type a question:

“Why is the sky blue?”

The model:

1. Converts your text into tokens  
2. Embeds the tokens  
3. Passes them through every transformer layer  
4. Produces a hidden representation  
5. Predicts the next token  
6. Samples that token  
7. Appends it to the input  
8. Repeats steps 3–7 until the answer is complete  

It’s literally **predicting one token at a time**, but the stacked layers + training give it the ability to:

- reason  
- explain  
- write  
- code  
- follow logic  

---

# So how does your layer fit into all this?

Your one encoder block is the **fundamental unit**.

Every major model you’ve heard of is just:

Transformers stacked + trained at massive scale.

By understanding a single layer, you understand *the core mechanism* behind:

- GPT models  
- ChatGPT  
- LLaMA  
- PaLM  
- T5  
- ViT  

ChatGPT is not magic.  
It is your encoder layer → repeated → scaled → trained → aligned.

That’s it.